# SLIIT IT4010 Research Project 2026 (Group J26-IT-333)
## Component 4: AI & IoT-Based Smart Waste Bin - Food Waste Recognition
### YOLO Pilot Training & Evaluation Notebook (Google Colab GPU Edition)

This notebook trains and evaluates the lightweight YOLO object detection pilot for food waste recognition across **7 active classes**:
1. `rice`
2. `kottu`
3. `noodles`
4. `chicken`
5. `whole_mango`
6. `mango_flesh`
7. `mango_peel`

> **Academic Methodology Note**:
> - In accordance with research integrity requirements, multi-photo capture sequences (e.g. `chicken_plate_001`–`020`, `mango_peel_wm_001`–`002` + `add_002`–`005`) are kept strictly isolated in the **Train** split to eliminate cross-split visual leakage.
> - Consequently, `chicken` and `mango_peel` have 0 evaluation samples in the initial pilot test split. Their mAP cannot be evaluated until independent unseen batches are captured.

### 1. GPU Acceleration Verification
Ensure your Google Colab runtime is set to GPU (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
# Verify NVIDIA GPU access
!nvidia-smi
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")

### 2. Install Required Dependencies
Install the official Ultralytics framework and utility libraries.

In [ ]:
%pip install ultralytics pyyaml pillow matplotlib

### 3. Upload & Unpack the Pilot Dataset (`food_waste_pilot_v2.zip`)
Upload `food_waste_pilot_v2.zip` from your local repo (`backend/ai/datasets/food_waste_pilot_v2.zip`) or Google Drive.

In [ ]:
import os
import shutil
import zipfile
import yaml
from pathlib import Path

DATASET_ZIP = "food_waste_pilot_v2.zip"
TARGET_DIR = Path("/content/food_waste_pilot_v2")

if not Path(DATASET_ZIP).exists():
    print(f"[*] Please upload '{DATASET_ZIP}' via the Files tab on the left sidebar...")
    from google.colab import files
    uploaded = files.upload()

TARGET_DIR.mkdir(parents=True, exist_ok=True)
print(f"[*] Extracting {DATASET_ZIP} to {TARGET_DIR}...")
with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    zf.extractall(TARGET_DIR)

# Update data.yaml with absolute Colab path
yaml_file = TARGET_DIR / "data.yaml"
with open(yaml_file, "r") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = str(TARGET_DIR)
data_cfg["train"] = "images/train"
data_cfg["val"] = "images/val"
data_cfg["test"] = "images/test"

with open(yaml_file, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"[+] Dataset ready at: {TARGET_DIR}")
print(f"[+] Configured classes: {data_cfg['names']}")

### 4. Verify Dataset Split Counts & Integrity

In [ ]:
for split in ["train", "val", "test"]:
    img_count = len(list((TARGET_DIR / "images" / split).glob("*")))
    lbl_count = len(list((TARGET_DIR / "labels" / split).glob("*")))
    print(f"{split.upper():6s} split: {img_count:3d} images, {lbl_count:3d} labels")

### 5. Launch YOLO Pilot Training
Using pretrained `yolov8n.pt` weights:
- Image size: 640
- Target Epochs: 100
- Early Stopping Patience: 20
- Batch size: 16 (on GPU)
- Seed: 42 (reproducibility)
- Augmentation applied strictly to train split

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(TARGET_DIR / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/content/runs/train",
    name="food-waste-yolo-pilot-v2-colab",
    save=True,
    verbose=True
)

best_weights = Path("/content/runs/train/food-waste-yolo-pilot-v2-colab/weights/best.pt")
print(f"[+] Training complete. Best weights saved at: {best_weights}")

### 6. Evaluate Model on Held-Out Test Split
The test split was kept strictly untouched during training and model selection.

In [ ]:
test_eval = model.val(
    data=str(TARGET_DIR / "data.yaml"),
    split="test",
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/content/runs/val",
    name="test_eval"
)

metrics = test_eval.box
print("=" * 60)
print("HELD-OUT TEST SET EVALUATION RESULTS")
print("=" * 60)
print(f"Overall Precision (P) : {metrics.mp:.2%}")
print(f"Overall Recall (R)    : {metrics.mr:.2%}")
print(f"mAP@0.5               : {metrics.map50:.2%}")
print(f"mAP@0.5:0.95          : {metrics.map:.2%}")
print("-" * 60)

ap_class_index = list(metrics.ap_class_index) if hasattr(metrics, "ap_class_index") else []
cid_to_idx = {int(cid): i for i, cid in enumerate(ap_class_index)}

print("PER-CLASS PERFORMANCE:")
for cid, cname in sorted(test_eval.names.items()):
    if cid in cid_to_idx:
        i = cid_to_idx[cid]
        p = metrics.p[i] if len(metrics.p) > i else 0.0
        r = metrics.r[i] if len(metrics.r) > i else 0.0
        ap50 = metrics.ap50[i] if len(metrics.ap50) > i else 0.0
        print(f"  • {cname:12s}: P={p:.2%}, R={r:.2%}, mAP50={ap50:.2%}")
    else:
        print(f"  • {cname:12s}: UNASSESSED (0 test samples in split; isolated in Train)")
print("=" * 60)

### 7. Visualize Confusion Matrix and Loss Curves

In [ ]:
from IPython.display import Image, display

run_dir = Path("/content/runs/train/food-waste-yolo-pilot-v2-colab")
for plot_name in ["results.png", "confusion_matrix.png", "PR_curve.png", "val_batch0_pred.jpg"]:
    p = run_dir / plot_name
    if p.exists():
        print(f"Displaying {plot_name}:")
        display(Image(filename=str(p)))
    else:
        print(f"Plot {plot_name} not found.")

### 8. Download Trained Model Weights (`best.pt`)
Download the best weights to save back to `Research-/backend/ai/models/best.pt`.

In [ ]:
from google.colab import files
if best_weights.exists():
    print(f"Downloading {best_weights.name}...")
    files.download(str(best_weights))
else:
    print("best.pt not found!")